AI Career Mentor

A beginner-friendly LLM-powered career guidance system built using Python and the OpenRouter API.

This project extracts student information, generates personalized career roadmaps, recommends projects, and stores the results in a structured JSON report.

Project Overvuew

Many students know their target career role but are often unsure about what skills, projects, and learning roadmap they should follow.

This project uses a Large Language Model (LLM) to:

- Extract student information from unstructured text
- Convert the information into structured JSON format
- Generate a personalized learning roadmap
- Suggest beginner-friendly projects
- Maintain conversation history
- Store the generated guidance in a JSON report

The goal of this project is to demonstrate prompt engineering, API integration, JSON processing, and conversational AI concepts using Python.

Features

- Student profile analysis
- Information extraction using LLM
- Structured JSON generation
- Career roadmap generation
- Beginner project recommendations
- Conversation memory support
- JSON report creation and storage
- Prompt engineering workflow

 Workflow

1. Student enters profile information.
2. The profile is sent to the LLM.
3. The LLM extracts structured student data.
4. JSON data is validated and parsed.
5. A Career Mentor prompt is created.
6. The mentor generates a learning roadmap.
7. The mentor suggests beginner projects.
8. Results are stored in a career report.
9. The report is saved as a JSON file.

 Technologies Used

 Programming Language
- Python

 Libraries
- requests
- json
- re

 AI Model
- DeepSeek V4 Flash

 API Provider
- OpenRouter API

 Development Environment
- Google Colab

 Data Format
- JSON

In [2]:
!pip install requests -q


In [3]:
import requests
import json
import re
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
from google.colab import userdata

API_KEY = userdata.get("Key-one")

API_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

MODEL = "deepseek/deepseek-v4-flash"

print("API configuration loaded successfully.")

API configuration loaded successfully.


In [5]:
def build_prompt(
    role,
    task,
    context,
    example_input,
    example_output,
    max_words
):
    return f"""
You are a {role}.

Task:
{task}

Context:
{context}

Example:

Input:
{example_input}

Output:
{example_output}

Constraints:

1. Maximum {max_words} words
2. Use bullet points
3. Use simple language
4. Avoid unnecessary technical terms
"""

print("Prompt builder loaded successfully.")

Prompt builder loaded successfully.


In [7]:
def call_llm(
    user_message,
    system_message="You are a helpful assistant.",
    max_tokens=300,
    temperature=0.7
):

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        "max_tokens": max_tokens,
        "temperature": temperature
    }

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload
    )

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"]

print("LLM API function loaded successfully.")

LLM API function loaded successfully.


In [8]:
response = call_llm(
    user_message="Discribe rose in  one sentence."
)

print(response)

A rose is a timeless symbol of love and beauty, characterized by its delicate, layered petals and enchanting fragrance, often adorned with thorns along its stem.


In [10]:
def safe_parse_json(text):

    text = text.strip()

    try:
        return json.loads(text)

    except json.JSONDecodeError:

        fenced = re.search(
            r'\{.*\}',
            text,
            re.DOTALL
        )

        if fenced:
            return json.loads(
                fenced.group(0)
            )

        raise

print("Safe JSON parser loaded successfully.")

Safe JSON parser loaded successfully.


In [11]:
def chat_with_history(chat_history, user_message):

    chat_history.append(
        {
            "role": "user",
            "content": user_message
        }
    )

    payload = {
        "model": MODEL,
        "messages": chat_history,
        "max_tokens": 300,
        "temperature": 1.0
    }

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload
    )

    response.raise_for_status()

    data = response.json()

    bot_reply = data["choices"][0]["message"]["content"]

    chat_history.append(
        {
            "role": "assistant",
            "content": bot_reply
        }
    )

    return chat_history, bot_reply

print("Chat memory function loaded successfully.")

Chat memory function loaded successfully.


In [12]:
chat_history = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    }
]

chat_history, reply = chat_with_history(
    chat_history,
    "My name is Keerthana."
)

print(reply)

chat_history, reply = chat_with_history(
    chat_history,
    "What is my name?"
)

print(reply)

Hello Keerthana! It's nice to meet you. How can I assist you today?
Your name is Keerthana.


In [13]:
student_info = """
Name: Keerthana
Target Role: AI Engineer

Current Skills:
Python
NumPy
HTML
"""

print(student_info)


Name: Keerthana
Target Role: AI Engineer

Current Skills:
Python
NumPy
HTML



In [14]:
print(type(student_info))

<class 'str'>


In [16]:
json_prompt = build_prompt(
    role="Information Extraction Assistant",

    task="""
Extract the student information and return ONLY valid JSON.

{
    "name":"",
    "target_role":"",
    "skills":[]
}
""",

    context="Student Profile",

    example_input="""
Name: Ravi
Target Role: Data Scientist
Skills: Python, SQL
""",

    example_output='{"name":"Ravi","target_role":"Data Scientist","skills":["Python","SQL"]}',

    max_words=100
)

In [17]:
response = call_llm(
    user_message=student_info,
    system_message=json_prompt,
    temperature=0.0
)

print(response)

{"name":"Keerthana","target_role":"AI Engineer","skills":["Python","NumPy","HTML"]}


In [18]:
student_data = safe_parse_json(response)

print("EXTRACTED STUDENT DATA")
print(student_data)

EXTRACTED STUDENT DATA
{'name': 'Keerthana', 'target_role': 'AI Engineer', 'skills': ['Python', 'NumPy', 'HTML']}


In [19]:
mentor_prompt = build_prompt(
    role="Career Mentor",

    task="Guide the student towards becoming an AI Engineer",

    context=str(student_data),

    example_input="What should I learn next?",

    example_output="""
• Learn SQL
• Learn Git
• Learn Data Structures
""",

    max_words=150
)

print(mentor_prompt)


You are a Career Mentor.

Task:
Guide the student towards becoming an AI Engineer

Context:
{'name': 'Keerthana', 'target_role': 'AI Engineer', 'skills': ['Python', 'NumPy', 'HTML']}

Example:

Input:
What should I learn next?

Output:

• Learn SQL
• Learn Git
• Learn Data Structures


Constraints:

1. Maximum 150 words
2. Use bullet points
3. Use simple language
4. Avoid unnecessary technical terms



In [20]:
chat_history = [
    {
        "role": "system",
        "content": mentor_prompt
    }
]

print("Career mentor initialized.")

Career mentor initialized.


In [21]:
print(chat_history)

[{'role': 'system', 'content': "\nYou are a Career Mentor.\n\nTask:\nGuide the student towards becoming an AI Engineer\n\nContext:\n{'name': 'Keerthana', 'target_role': 'AI Engineer', 'skills': ['Python', 'NumPy', 'HTML']}\n\nExample:\n\nInput:\nWhat should I learn next?\n\nOutput:\n\n• Learn SQL\n• Learn Git\n• Learn Data Structures\n\n\nConstraints:\n\n1. Maximum 150 words\n2. Use bullet points\n3. Use simple language\n4. Avoid unnecessary technical terms\n"}]


In [22]:
chat_history, reply = chat_with_history(
    chat_history,
    "What skills should I learn next?"
)

print("SKILLS TO LEARN")
print(reply)

SKILLS TO LEARN
Great question, Keerthana! To become an AI Engineer, you’ll need to build on your Python and NumPy foundation. Here are the next skills to focus on:

- **Learn Pandas** – for handling and cleaning real-world data (like spreadsheets).
- **Learn SQL** – to fetch and manage data from databases.
- **Learn Git** – to save and share your code with others (essential for teamwork).
- **Learn basic Machine Learning concepts** – start with simple models (like linear regression) using Scikit-learn.
- **Practice with a project** – combine Python, NumPy, Pandas, and ML on a small dataset (e.g., predicting house prices).

These steps will give you a strong start. After you master these, you can dive into deep learning and AI frameworks. You’re on the right track!


In [23]:
chat_history, reply = chat_with_history(
    chat_history,
    "Create a 3 month roadmap."
)

print("3 MONTH ROADMAP")
print(reply)

3 MONTH ROADMAP
Here's a simple 3-month roadmap to become an AI Engineer. Focus on one step at a time.

**Month 1: Data & Databases**
- Learn Pandas (focus on reading CSV files, cleaning data, simple analysis)
- Learn SQL (basic queries like SELECT, WHERE, JOIN, GROUP BY)
- Learn Git (basic commands: add, commit, push, pull)

**Month 2: Machine Learning Basics**
- Learn Machine Learning concepts (supervised vs unsupervised learning)
- Learn Scikit-learn library (train/test split, linear regression, decision trees)
- Build one small project (e.g., predict student grades using a dataset)

**Month 3: Deep Learning Start & Portfolio**
- Learn basic deep learning (neurons, layers, activation functions)
- Try Keras/TensorFlow (build a simple neural network)
- Upload your projects to GitHub
- Apply for internships or junior AI roles

Remember: practice daily, even 30 minutes. You've got this, Keerthana!


In [24]:
chat_history, reply = chat_with_history(
    chat_history,
    "Suggest 3 beginner projects."
)

print("PROJECT IDEAS")
print(reply)

PROJECT IDEAS
Here are 3 beginner-friendly projects to build your AI portfolio, Keerthana:

- **House Price Predictor** – Use a dataset (like Boston Housing) with Pandas and Scikit-learn. Train a simple linear regression model to predict prices. Great for learning data cleaning and model training.

- **Movie Recommender** – Use a small movie dataset and NumPy/Pandas to recommend movies based on similarity (e.g., genre or rating). No machine learning needed – just basic math and data filtering.

- **Spam Email Classifier** – Use a text dataset (e.g., SMS Spam Collection) and Scikit-learn’s Naive Bayes. Learn to convert text into numbers and build a classifier. Fun and practical.

Pick one, finish it, then move to the next. Upload each to


In [25]:
career_report = {
    "student_name": student_data["name"],
    "target_role": student_data["target_role"],
    "current_skills": student_data["skills"],
    "recommended_roadmap": "",
    "recommended_projects": ""
}

print(career_report)

{'student_name': 'Keerthana', 'target_role': 'AI Engineer', 'current_skills': ['Python', 'NumPy', 'HTML'], 'recommended_roadmap': '', 'recommended_projects': ''}


In [26]:
chat_history, roadmap = chat_with_history(
    chat_history,
    "Create a detailed 3 month roadmap."
)

career_report["recommended_roadmap"] = roadmap

print("ROADMAP SAVED")

ROADMAP SAVED


In [27]:
chat_history, projects = chat_with_history(
    chat_history,
    "Suggest 5 beginner projects."
)

career_report["recommended_projects"] = projects

print("PROJECTS SAVED")

PROJECTS SAVED


In [28]:
print(career_report)

{'student_name': 'Keerthana', 'target_role': 'AI Engineer', 'current_skills': ['Python', 'NumPy', 'HTML'], 'recommended_roadmap': "**Month 1: Data Handling & Tools**\n- Week 1: Learn Pandas basics (read CSV, select columns, filter rows)\n- Week 2: Learn Pandas advanced (groupby, merge, handle missing data)\n- Week 3: Learn SQL (SELECT, WHERE, JOIN, GROUP BY) – practice on a small database\n- Week 4: Learn Git (commit, push, pull, branches) and create a GitHub account\n\n**Month 2: Machine Learning Basics**\n- Week 1: Understand supervised learning (regression, classification) and train/test split\n- Week 2: Build linear regression model with Scikit-learn on a dataset\n- Week 3: Build a decision tree classifier (e.g., iris dataset) and check accuracy\n- Week 4: Complete project: House Price Predictor using Pandas + Scikit-learn, upload to GitHub\n\n**Month 3: Deep Learning & Portfolio**\n- Week 1: Learn basic neural network concepts (neurons, layers, activation functions)\n- Week 2: Bui